# Leg-2 encoder pilot

Relabel a lane with a complete alternative retrieval stack and diff against the shipped labels. The default Leg-2 stack here is **Qwen3 dense + SPLADE sparse**; RRF is rebuilt from those two legs. Row identity, qrels, and the scoring objective stay fixed, so movement is attributable to the stack swap.

Every query falls in exactly one of three buckets, decided by its three route scores:

| bucket | meaning |
| --- | --- |
| **`all_zero`** | all three scores are 0 — nothing relevant found |
| **`all_tied`** | all three scores are the same — every route did equally well |
| **`routes_differ`** | anything else — the routes disagree |

That is `labels.outcome_shape`, the repo's own definition. One section per bucket: **where did those rows go, and what moved in the scores.**

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os, sys, pathlib

SRC = pathlib.Path.cwd()
SRC = SRC if (SRC / 'scripts').exists() else SRC / 'src'
sys.path.insert(0, str(SRC))

import pandas as pd
from dotenv import load_dotenv
from qdrant_client import QdrantClient

from composition.pool_v3 import SCORES
from scripts.legb import (
    LEGB_DIR, LegBPilot, e5_dense_cfg, qwen_dense_cfg, splade_sparse_cfg,
)

load_dotenv()
client = QdrantClient(
    url=os.environ['QDRANT_CLOUD_URL'],
    api_key=os.environ['QDRANT_CLOUD_API_KEY'],
    timeout=120,
    cloud_inference=True,
)
pd.set_option('display.width', 170)

/Users/andrei/projects/hybrid-search-rrf-dataset/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. What to draw

One entry per bucket, all labelled in a single pass below. Comment one out to skip it — its section then says `not sampled yet`.

`n` is **per lane**, fixed seed.

In [3]:
DENSE_ENCODER  = qwen_dense_cfg   # or e5_dense_cfg (local, free, no API key)
SPARSE_ENCODER = splade_sparse_cfg
OUT_DIR = LEGB_DIR / 'qwen3_splade'  # never reuse the earlier Qwen+BM25 labels
LANE = 'crumb-legal-qa'  # None = every pilot lane

DRAWS = {
    'all_tied':      ('all_tied', 60),
    'all_zero':      ('all_zero', 100),
    'routes_differ': ('routes_differ', 100),
}

pilots = {name: LegBPilot(
              client, DENSE_ENCODER(), sparse_cfg=SPARSE_ENCODER(),
              out_dir=OUT_DIR, sample=draw,
          )
          for name, draw in DRAWS.items()}

pd.concat([p.plan().assign(draw=name) for name, p in pilots.items()],
          ignore_index=True)[['draw', 'lane', 'to_label', 'indexed']]

[antique] sample asked 60 all_tied rows, drew 0 — that is the whole available population
[antique] sample asked 100 all_zero rows, drew 0 — that is the whole available population
[antique] sample asked 100 routes_differ rows, drew 0 — that is the whole available population


,draw,lane,to_label,indexed
0,all_tied,scirgen-geo-en,60,False
1,all_tied,crumb-legal-qa,60,False
2,all_tied,antique,0,False
3,all_zero,scirgen-geo-en,100,False
4,all_zero,crumb-legal-qa,100,False
5,all_zero,antique,0,False
6,routes_differ,scirgen-geo-en,100,False
7,routes_differ,crumb-legal-qa,100,False
8,routes_differ,antique,0,False


## 2. Label

Idempotent — a full collection skips indexing, an already-labelled query is not rescored. This is the paid step.

In [4]:
for name, p in pilots.items():
    print(f'\n########## {name} ##########')
    failed = p.index_and_label(lane=LANE)
    if failed:
        print('  FAILED:', failed)


########## all_tied ##########
=== [1/1] crumb-legal-qa (openrouter/qwen/qwen3-embedding-8b, max_workers=8) ===
[crumb-legal-qa] indexing 10,000/10,000 -> crumb-legal-qa_legb_qwen3-embedding-8b_Splade_PP_en_v1_routes


embed:sparse_legb: 100%|██████████| 10000/10000 [11:11<00:00, 14.90it/s]
upload:crumb-legal-qa_legb_qwen3-embedding-8b_Splade_PP_en_v1_routes: 100%|██████████| 157/157 [12:04<00:00,  4.62s/it]
label:crumb-legal-qa: 100%|██████████| 1/1 [00:23<00:00, 23.87s/chunk, differ=41, labelled=60, tied=19, zero=0]


[crumb-legal-qa] +60 leg-2 labels  differ 41 | tied 19 | zero 0

########## all_zero ##########
=== [1/1] crumb-legal-qa (openrouter/qwen/qwen3-embedding-8b, max_workers=8) ===


label:crumb-legal-qa: 100%|██████████| 1/1 [00:38<00:00, 38.91s/chunk, differ=38, labelled=100, tied=1, zero=61]


[crumb-legal-qa] +100 leg-2 labels  differ 38 | tied 1 | zero 61

########## routes_differ ##########
=== [1/1] crumb-legal-qa (openrouter/qwen/qwen3-embedding-8b, max_workers=8) ===


label:crumb-legal-qa: 100%|██████████| 1/1 [00:38<00:00, 38.50s/chunk, differ=82, labelled=100, tied=12, zero=6]

[crumb-legal-qa] +100 leg-2 labels  differ 82 | tied 12 | zero 6


## 3. Pair the legs

`paired` = one row per query, both legs' three scores side by side, plus each leg's bucket and winning route.

Both retrieval arms should now move. The controls are the exact same rows, qrels, objective, and scoring regime. A zero sparse-change count is treated as a wiring or stale-output failure, because this run is specifically meant to exercise SPLADE.

In [5]:
SHORT = {'score_dense_only': 'dense', 'score_sparse_only': 'sparse',
         'score_pure_rrf': 'rrf'}

pilot = next(iter(pilots.values()))
leg2 = pd.read_parquet(OUT_DIR / 'labels.parquet')[['dataset', 'query_id'] + SCORES]
leg1 = pilot._pool.labels()[['dataset', 'query_id'] + SCORES]
paired = leg1.merge(leg2, on=['dataset', 'query_id'], suffixes=('_1', '_2'))

for leg in ('1', '2'):
    cols = [c + '_' + leg for c in SCORES]
    paired['max_' + leg] = paired[cols].max(axis=1)
    paired['min_' + leg] = paired[cols].min(axis=1)
    # the repo's own three-way split: all zero / all the same / they differ
    paired['bucket_' + leg] = 'routes_differ'
    paired.loc[paired['max_' + leg] - paired['min_' + leg] <= 1e-9,
               'bucket_' + leg] = 'all_tied'
    paired.loc[paired['max_' + leg] <= 1e-9, 'bucket_' + leg] = 'all_zero'
    paired['winner_' + leg] = paired[cols].idxmax(axis=1).map(
        lambda c: SHORT[c.rsplit('_', 1)[0]])

changed_by_route = {}
for col, short in SHORT.items():
    changed = int((paired[col + '_1'].round(9) != paired[col + '_2'].round(9)).sum())
    changed_by_route[short] = changed
    print(f'{short:7} changed on {changed:>4}/{len(paired)}')

if len(paired) and changed_by_route['sparse'] == 0:
    raise AssertionError('SPLADE produced zero sparse-score changes; check OUT_DIR/config wiring')

print(f'\n{len(paired)} rows paired')
paired['bucket_1'].value_counts().rename('drawn from')

dense   changed on  147/260
sparse  changed on  112/260
rrf     changed on  139/260

260 rows paired


bucket_1
routes_differ    100
all_zero         100
all_tied          60
Name: drawn from, dtype: int64

### Where everything went

The whole result in one table: leg-1 bucket down the side, leg-2 bucket across. The diagonal stayed put.

In [6]:
pd.crosstab(paired['bucket_1'], paired['bucket_2'], margins=True)

bucket_2,all_tied,all_zero,routes_differ,All
bucket_1,,,,
all_tied,19,0,41,60
all_zero,1,61,38,100
routes_differ,12,6,82,100
All,32,67,161,260


### The one number: did retrieval find the judged document?

Ignore labels and margins for a second. Each query has a judged relevant document. Either some route retrieved it (score > 0) or none did (all zero). That is the retrieval stack's actual job, and it is the cleanest comparison available:

- **restored** — leg-1 found nothing, leg-2 found it
- **destroyed** — leg-1 found it, leg-2 lost it

Everything downstream (which route wins, whether the margin certifies) is labelling *policy*. This is the encoder.

In [7]:
restored = paired[(paired.max_1 <= 1e-9) & (paired.max_2 > 1e-9)]
destroyed = paired[(paired.max_1 > 1e-9) & (paired.max_2 <= 1e-9)]
print(f'restored  (nothing -> found) : {len(restored)}')
print(f'destroyed (found -> nothing) : {len(destroyed)}')
print(f'NET on information           : {len(restored) - len(destroyed):+d}')

for col, short in SHORT.items():
    delta = paired[col + '_2'] - paired[col + '_1']
    print(f'\n{short} moved up/down/unchanged: ' 
          f'{int((delta > 1e-9).sum())}/'
          f'{int((delta < -1e-9).sum())}/'
          f'{int((delta.abs() <= 1e-9).sum())}')

restored  (nothing -> found) : 39
destroyed (found -> nothing) : 6
NET on information           : +33

dense moved up/down/unchanged: 103/44/113

sparse moved up/down/unchanged: 69/43/148

rrf moved up/down/unchanged: 110/29/121


## Differentiator gate

Formalises "is this stack a net-positive labeller?" as three **signed pass conditions**, not raw movement: coverage split by headroom (`all_zero` ≥25%, low ties break up ≥10%, ceiling ties break ≤5%), a benefit-vs-damage ratio ≥3:1, and a **decisiveness gain** — the candidate must raise the certifiable-decisive share (margin ≥ 0.1) *above the baseline*, or it fails. That last one stops a stack from "passing" by shoving `all_zero` rows into ties without ever producing a label. The `tied_by_regression` rows above are exactly the signed damage the ratio catches. Shared module `scripts/differentiator_gate.py`; R1 runs the same gate at `trust="opinion"`.

In [ ]:
from scripts.differentiator_gate import differentiator_gate

# Leg2 is trust="verified": its scores run against the real qrels, so an all_zero
# that moves off zero genuinely retrieved the human-judged doc, and a broken
# ceiling tie genuinely means a route regressed (not an over-accepted extra doc).
differentiator_gate(paired, base="1", cand="2", trust="verified").report()

---
# Eye test — real queries, per case

Six cases. `cases()` lists them with counts; `eye(name, n)` prints the queries with both legs' scores so you can judge whether the movement makes sense.

Read the score triples as `dense / sparse / rrf`. In this complete stack swap, both dense and sparse can move; the printed deltas identify which arm caused a bucket or winner change.

In [8]:
import textwrap

# query text lives in the leg-2 labels file
_q = pd.read_parquet(OUT_DIR / 'labels.parquet')[['dataset', 'query_id', 'query']]
eyes = paired.merge(_q, on=['dataset', 'query_id'], how='left')
eyes['margin'] = eyes.max_2 - eyes[[c + '_2' for c in SCORES]].apply(
    lambda r: sorted(r)[-2], axis=1)
eyes['dd'] = eyes.score_dense_only_2 - eyes.score_dense_only_1
eyes['ds'] = eyes.score_sparse_only_2 - eyes.score_sparse_only_1

_tied = eyes.bucket_1 == 'all_tied'
_zero = eyes.bucket_1 == 'all_zero'
_broke = _tied & (eyes.bucket_2 == 'routes_differ')

CASES = {
    'zero_to_decisive':  (_zero & (eyes.bucket_2 == 'routes_differ') & (eyes.margin >= 0.4),
                          'all_zero -> found it, decisively (margin >= 0.4)'),
    'zero_to_nothing':   (_zero & (eyes.bucket_2 == 'all_zero'),
                          'all_zero -> still nothing (no encoder fixes these)'),
    'tied_to_decisive':  (_broke & (eyes.margin >= 0.4),
                          'all_tied -> broke decisively (margin >= 0.4)'),
    'tied_to_thin':      (_broke & (eyes.margin < 0.1),
                          'all_tied -> broke thin (margin < 0.1)'),
    'tied_by_regression': (_broke & (eyes.dd < -1e-9) & (eyes.ds <= 1e-9),
                           'all_tied -> broke with dense down and no sparse gain'),
    'destroyed':         ((eyes.max_1 > 1e-9) & (eyes.max_2 <= 1e-9),
                          'had a hit -> lost it entirely'),
}


def cases():
    return pd.DataFrame([{'case': k, 'rows': int(m.sum()), 'what': d}
                         for k, (m, d) in CASES.items()]).set_index('case')


def eye(case, n=5, sort='margin'):
    """Queries for one case, with both legs' scores as dense/sparse/rrf.

    Prints the query in FULL, wrapped. Several lanes are templated --
    crumb-legal-qa carries a 114-char boilerplate preamble ("Secondary methods
    of service are defined as...") -- so any head truncation cuts before the
    words that distinguish the rows and makes four different queries look like
    one repeated row. Nothing is hidden here for that reason.
    """
    mask, desc = CASES[case]
    rows = eyes[mask].sort_values(sort, ascending=False).head(n)
    print(f'{case} — {desc}\n{int(mask.sum())} rows, showing {len(rows)}\n' + '=' * 96)
    if rows.empty:
        print('  (none)')
        return None
    for _, r in rows.iterrows():
        print(textwrap.fill(str(r['query']), width=94,
                            initial_indent=f'[{r.query_id}] ', subsequent_indent=' ' * 8))
        print(f"   leg-1   {r.score_dense_only_1:.3f} / {r.score_sparse_only_1:.3f} / {r.score_pure_rrf_1:.3f}")
        print(f"   leg-2   {r.score_dense_only_2:.3f} / {r.score_sparse_only_2:.3f} / {r.score_pure_rrf_2:.3f}"
              f"   -> {r.winner_2}, margin {r.margin:.3f}, "
              f"dense {r.dd:+.3f}, sparse {r.ds:+.3f}\n")
    return None


cases()

,rows,what
case,,
zero_to_decisive,4,"all_zero -> found it, decisively (margin >= 0.4)"
zero_to_nothing,61,all_zero -> still nothing (no encoder fixes th...
tied_to_decisive,8,all_tied -> broke decisively (margin >= 0.4)
tied_to_thin,33,all_tied -> broke thin (margin < 0.1)
tied_by_regression,14,all_tied -> broke with dense down and no spars...
destroyed,6,had a hit -> lost it entirely


### `zero_to_decisive`

The clean wins. Dense goes 0 -> 1.0: Qwen puts the judged doc at rank 1 where bge-small had nothing. Sparse stays 0, so BM25 never had these either.

In [9]:
eye('zero_to_decisive', n=5)

zero_to_decisive — all_zero -> found it, decisively (margin >= 0.4)
4 rows, showing 4
[5697] Secondary methods of service are defined as those methods that may be used if the
        primary method is unsuccessful. Is personal service to suitable person other than
        defendant a permitted secondary method of service for an eviction action? In the state
        of Nevada
   leg-1   0.000 / 0.000 / 0.000
   leg-2   1.000 / 0.000 / 0.189   -> dense, margin 0.811, dense +1.000, sparse +0.000

[38] Is there a state/territory law regulating residential evictions? In the state of Ohio
   leg-1   0.000 / 0.000 / 0.000
   leg-2   1.000 / 0.000 / 0.189   -> dense, margin 0.811, dense +1.000, sparse +0.000

[4000] Are eviction cases first heard in magistrates court? In the state of Colorado
   leg-1   0.000 / 0.000 / 0.000
   leg-2   0.976 / 0.000 / 0.187   -> dense, margin 0.789, dense +0.976, sparse +0.000

[7802] Is the term 'Warrant of ejectment' used to refer to the order from the court

### `zero_to_nothing`

Both encoders fail identically. Look at the phrasing - these are near-duplicate templated queries. No dense model fixes this; it is what leg-3 (the LLM judge) exists for.

In [10]:
eye('zero_to_nothing', n=5)

zero_to_nothing — all_zero -> still nothing (no encoder fixes these)
61 rows, showing 5
[7957] Is the term 'Writ of eviction' used to refer to the order from the court to the
        authorities to remove a tenant? In the state of Massachusetts
   leg-1   0.000 / 0.000 / 0.000
   leg-2   0.000 / 0.000 / 0.000   -> dense, margin 0.000, dense +0.000, sparse +0.000

[8408] Is it specified what term is used to refer to the order from the court to the
        authorities to remove a tenant? In the state of South Dakota
   leg-1   0.000 / 0.000 / 0.000
   leg-2   0.000 / 0.000 / 0.000   -> dense, margin 0.000, dense +0.000, sparse +0.000

[1134] Can a landlord evict a tenant for nuisance activity? This includes: maintaining,
        committing, or permitting the maintenance or commission of a nuisance; conduct that
        disturbs a neighbor’s or another tenant’s peaceful enjoyment of their premises;
        causing a serious health hazard to exist on the property; or for misuse of property

### `tied_to_decisive`

Careful: two opposite things share this bucket. A row tied LOW that jumps to 1.0 is a win. A row tied at 1.0 whose dense collapses is a regression with an identical margin. Check the dense delta sign.

In [11]:
eye('tied_to_decisive', n=5)

tied_to_decisive — all_tied -> broke decisively (margin >= 0.4)
8 rows, showing 5
[5332] Secondary methods of service are defined as those methods that may be used if the
        primary method is unsuccessful. Is publication a permitted secondary method of service
        for an eviction action? In the state of Arkansas
   leg-1   1.000 / 1.000 / 1.000
   leg-2   0.189 / 0.129 / 1.000   -> rrf, margin 0.811, dense -0.811, sparse -0.871

[4327] Are eviction cases first heard in district court? In the state of New Hampshire
   leg-1   1.000 / 1.000 / 1.000
   leg-2   1.000 / 0.116 / 0.189   -> dense, margin 0.811, dense +0.000, sparse -0.884

[8376] Is the term 'Execution' used to refer to the order from the court to the authorities to
        remove a tenant? In the state of Rhode Island
   leg-1   1.000 / 1.000 / 1.000
   leg-2   0.150 / 1.000 / 0.189   -> sparse, margin 0.811, dense -0.850, sparse +0.000

[7444] When a tenant appeals, is the execution of eviction stayed for the durat

### `tied_to_thin`

Dense improved and won, narrowly. Sparse did not move, so the movement is real - just small. By the working bar these count.

In [12]:
eye('tied_to_thin', n=5)

tied_to_thin — all_tied -> broke thin (margin < 0.1)
33 rows, showing 5
[2327] Are fines assessed to landlords for unlawfully evicting a tenant? In the state of
        Georgia
   leg-1   0.884 / 0.884 / 0.884
   leg-2   0.163 / 0.884 / 0.955   -> rrf, margin 0.071, dense -0.721, sparse +0.000

[1664] Can a landlord evict a tenant for remaining on property after expiration of the lease?
        In the state of South Dakota
   leg-1   0.841 / 0.841 / 0.841
   leg-2   0.930 / 0.930 / 0.984   -> rrf, margin 0.054, dense +0.089, sparse +0.089

[3968] Are eviction cases first heard in county court? In the state of Arkansas
   leg-1   0.050 / 0.050 / 0.050
   leg-2   0.171 / 0.074 / 0.119   -> dense, margin 0.052, dense +0.121, sparse +0.023

[4572] Are eviction cases first heard in magistrates court? In the state of Virginia
   leg-1   1.000 / 1.000 / 1.000
   leg-2   0.189 / 0.107 / 0.150   -> dense, margin 0.039, dense -0.811, sparse -0.893

[2760] Is it unlawful to evict a tenant because

### `tied_by_regression`

The tie broke with dense getting worse and no compensating sparse gain. Never read these as the alternative stack helping.

In [13]:
eye(case='tied_by_regression', n=5)

tied_by_regression — all_tied -> broke with dense down and no sparse gain
14 rows, showing 5
[5332] Secondary methods of service are defined as those methods that may be used if the
        primary method is unsuccessful. Is publication a permitted secondary method of service
        for an eviction action? In the state of Arkansas
   leg-1   1.000 / 1.000 / 1.000
   leg-2   0.189 / 0.129 / 1.000   -> rrf, margin 0.811, dense -0.811, sparse -0.871

[8376] Is the term 'Execution' used to refer to the order from the court to the authorities to
        remove a tenant? In the state of Rhode Island
   leg-1   1.000 / 1.000 / 1.000
   leg-2   0.150 / 1.000 / 0.189   -> sparse, margin 0.811, dense -0.850, sparse +0.000

[5331] Secondary methods of service are defined as those methods that may be used if the
        primary method is unsuccessful. Is publication and mail used a permitted secondary
        method of service for an eviction action? In the state of Arkansas
   leg-1   1.000 / 1.

### `destroyed`

The cost side. These had a retrieved judged doc and now have none - labels lost outright.

In [14]:
eye('destroyed', n=5)

destroyed — had a hit -> lost it entirely
6 rows, showing 5
[7569] Is the term 'Writ of restitution' used to refer to the order from the court to the
        authorities to remove a tenant? In the state of Arkansas
   leg-1   0.107 / 0.000 / 0.000
   leg-2   0.000 / 0.000 / 0.000   -> dense, margin 0.000, dense -0.107, sparse +0.000

[1014] Can a landlord evict a tenant for removal of unit from market? This includes when the
        law allows a tenant to be evicted based on: the landlord selling the property;
        demolition, rehabilitation, or a change in use of the premises; repairs that would
        require the tenant to lose access to the property; a change to a policy of excluding
        children; or conversion of a rental property to a condominium, cooperative, or other
        form of ownership arrangement. In the state of Georgia
   leg-1   0.042 / 0.000 / 0.000
   leg-2   0.000 / 0.000 / 0.000   -> dense, margin 0.000, dense -0.042, sparse +0.000

[92] With regards to ev

---
## Aggregates per bucket

The same three buckets as counts rather than examples, if you want the totals after eyeballing the cases above. `show()` prints where one bucket's rows went plus their score table.

In [15]:
def show(bucket):
    """Rows drawn from one leg-1 bucket: where they went, and the scores."""
    rows = paired[paired.bucket_1 == bucket]
    if rows.empty:
        print(f'{bucket}: not sampled yet — add it to DRAWS and re-run cells 1-3')
        return None
    moved = rows[rows.bucket_1 != rows.bucket_2]
    print(f'{bucket}: {len(rows)} rows   stayed {len(rows) - len(moved)}   moved {len(moved)}')
    print(rows['bucket_2'].value_counts().to_string())
    cols = ['query_id'] + [c + s for c in ('dense', 'sparse', 'rrf') for s in ('_1', '_2')]
    out = rows.rename(columns={k + s: v + s for k, v in SHORT.items() for s in ('_1', '_2')})
    return (out[cols + ['bucket_2', 'winner_2']]
            .sort_values('dense_2', ascending=False).round(3))


print('helper ready')

helper ready


---
# A. `all_tied` — all three scores the same

Every route already did equally well. There are only two things that can happen:

- the scores stay level → **still tied**, the encoder changed nothing here;
- one route pulls apart from the others → **`routes_differ`**.

**Watch the direction.** If the tie was at the top (all three at 1.0, the judged doc already at rank 1) then dense cannot go *up* — so a tie that breaks there broke because dense got **worse**, and the row's new winner will be sparse. Compare `dense_1` to `dense_2` before reading a broken tie as an improvement.

In [16]:
show('all_tied')

all_tied: 60 rows   stayed 19   moved 41
bucket_2
routes_differ    41
all_tied         19


,query_id,dense_1,dense_2,sparse_1,sparse_2,rrf_1,rrf_2,bucket_2,winner_2
96,6346,1.000,1.000,1.000,0.095,1.000,1.000,routes_differ,dense
92,328,1.000,1.000,1.000,1.000,1.000,1.000,all_tied,dense
221,144,1.000,1.000,1.000,1.000,1.000,1.000,all_tied,dense
205,2341,1.000,1.000,1.000,0.189,1.000,1.000,routes_differ,dense
196,7653,0.129,1.000,0.129,1.000,0.129,1.000,all_tied,dense
111,3490,1.000,1.000,1.000,1.000,1.000,1.000,all_tied,dense
184,3104,0.087,1.000,0.087,0.000,0.087,1.000,routes_differ,dense
118,7444,0.189,1.000,0.189,0.189,0.189,0.189,routes_differ,dense
131,5860,1.000,1.000,1.000,1.000,1.000,1.000,all_tied,dense
133,9180,1.000,1.000,1.000,1.000,1.000,1.000,all_tied,dense


In [17]:
t = paired[paired.bucket_1 == 'all_tied']
if not t.empty:
    for col, short in SHORT.items():
        delta = t[col + '_2'] - t[col + '_1']
        print(f'{short:6} up/down/unchanged: ' 
              f'{int((delta > 1e-9).sum())}/'
              f'{int((delta < -1e-9).sum())}/'
              f'{int((delta.abs() <= 1e-9).sum())}')
    print('\nthe ties that broke were at what score level and who won?')
    broke = t[t.bucket_2 != 'all_tied']
    print(broke.assign(tied_at=broke.max_1.round(2))
          .groupby(['tied_at', 'winner_2']).size().rename('rows').to_string())

dense  up/down/unchanged: 16/15/29
sparse up/down/unchanged: 7/24/29
rrf    up/down/unchanged: 16/12/32

the ties that broke were at what score level and who won?
tied_at  winner_2
0.05     dense        1
0.07     dense        1
0.08     dense        1
0.09     dense        2
0.11     dense        1
0.12     dense        1
0.19     dense        3
         rrf          1
0.82     sparse       1
0.84     dense        1
         rrf          2
0.88     dense        6
         rrf          2
1.00     dense       10
         rrf          6
         sparse       2


---
# B. `all_zero` — all three scores are 0

No route surfaced anything relevant, even though a judged relevant document exists for the query. Nothing caps these: any score above 0 is an improvement.

**The question:** how many move off zero, how far, and which route class finds the judged document. A winning score `≥ 0.7` means a relevant document reached rank 1; below that it landed deeper in the top-10.

In [18]:
show('all_zero')

all_zero: 100 rows   stayed 61   moved 39
bucket_2
all_zero         61
routes_differ    38
all_tied          1


,query_id,dense_1,dense_2,sparse_1,sparse_2,rrf_1,rrf_2,bucket_2,winner_2
211,3441,0.0,1.000,0.0,0.15,0.0,1.000,routes_differ,dense
189,5697,0.0,1.000,0.0,0.00,0.0,0.189,routes_differ,dense
171,4350,0.0,1.000,0.0,0.00,0.0,1.000,routes_differ,dense
236,38,0.0,1.000,0.0,0.00,0.0,0.189,routes_differ,dense
202,4000,0.0,0.976,0.0,0.00,0.0,0.187,routes_differ,dense
...,...,...,...,...,...,...,...,...,...
82,7924,0.0,0.000,0.0,0.00,0.0,0.000,all_zero,dense
81,2391,0.0,0.000,0.0,0.00,0.0,0.000,all_zero,dense
79,771,0.0,0.000,0.0,0.00,0.0,0.000,all_zero,dense
78,7922,0.0,0.000,0.0,0.00,0.0,0.000,all_zero,dense


In [19]:
z = paired[paired.bucket_1 == 'all_zero']
if not z.empty:
    off = z[z.max_2 > 1e-9]
    print(f'moved off zero : {len(off)}/{len(z)}  ({len(off)/len(z):.0%})')
    if len(off):
        print(f'  judged doc at rank 1 : {int((off.max_2 >= 0.7).sum())}')
        print(f'  deeper in the top-10 : {int((off.max_2 < 0.7).sum())}')
        print('\nwhich route found it:')
        print(off['winner_2'].value_counts().to_string())
    if 'dataset' in z:
        print('\nper lane:')
        print(z.assign(off_zero=z.max_2 > 1e-9).groupby('dataset')['off_zero']
              .agg(rows='size', off_zero='sum').to_string())

moved off zero : 39/100  (39%)
  judged doc at rank 1 : 7
  deeper in the top-10 : 32

which route found it:
winner_2
dense     35
rrf        3
sparse     1

per lane:
                rows  off_zero
dataset                       
crumb-legal-qa   100        39


---
# C. `routes_differ` — the routes already disagree

These rows already say something about dense vs sparse, so the thing that matters is whether they now say something **different**.

**The question:** does the winning route change, and where does the flow go? With both arms replaced, movement toward dense, sparse, or RRF is meaningful; inspect the full transition matrix rather than expecting one direction.

In [20]:
show('routes_differ')

routes_differ: 100 rows   stayed 82   moved 18
bucket_2
routes_differ    82
all_tied         12
all_zero          6


,query_id,dense_1,dense_2,sparse_1,sparse_2,rrf_1,rrf_2,bucket_2,winner_2
0,296,0.116,1.0,0.116,0.150,0.129,1.000,routes_differ,dense
59,3417,0.129,1.0,0.000,0.087,0.095,0.189,routes_differ,dense
88,3842,0.189,1.0,0.129,0.090,0.189,1.000,routes_differ,dense
90,4431,0.116,1.0,0.000,1.000,0.090,1.000,all_tied,dense
93,5796,1.000,1.0,0.150,0.000,1.000,0.189,routes_differ,dense
...,...,...,...,...,...,...,...,...,...
203,7552,0.095,0.0,0.000,0.000,0.000,0.000,all_zero,dense
8,7569,0.107,0.0,0.000,0.000,0.000,0.000,all_zero,dense
200,7189,0.100,0.0,0.000,0.000,0.000,0.000,all_zero,dense
38,1014,0.042,0.0,0.000,0.000,0.000,0.000,all_zero,dense


In [21]:
d = paired[paired.bucket_1 == 'routes_differ']
if not d.empty:
    print('winning route, leg-1 (rows) x leg-2 (cols):')
    print(pd.crosstab(d['winner_1'], d['winner_2'], margins=True).to_string())
    flips = d[d.winner_1 != d.winner_2]
    print(f'\nwinner changed on {len(flips)}/{len(d)} rows')
    if len(flips):
        print('\nnew winners among flipped rows:')
        print(flips['winner_2'].value_counts().to_string())

winning route, leg-1 (rows) x leg-2 (cols):
winner_2  dense  rrf  sparse  All
winner_1                         
dense        73   11       2   86
rrf           7    1       1    9
sparse        5    0       0    5
All          85   12       3  100

winner changed on 26/100 rows

new winners among flipped rows:
winner_2
dense     12
rrf       11
sparse     3


---
## Reading it

- **`sparse` changes on zero rows** → stop; SPLADE was not exercised or stale output was loaded.
- **A: ties break with one arm down and no other arm up** → the alternative stack regressed; a broken tie is not automatically a win.
- **B: high off-zero rate with both dense and sparse winners** → the full stack adds useful route-class differentiation.
- **C: winner transitions** → read the matrix as stack sensitivity, including movement toward sparse and RRF.

One caveat that survives the simplification: `natural_only=True` by default, so `supplemented` rows (scored against a corpus carrying constructed documents) are excluded — a stack effect there would be tangled up with the augmentation. `antique` has no natural tied rows at all, so it cannot appear in section A. SPLADE is English-only, so do not run this configuration on non-English lanes.

To cross-check the dense arm, set `DENSE_ENCODER = e5_dense_cfg` and choose a new `OUT_DIR`. Keep SPLADE fixed if the question is dense robustness; change one component at a time for attribution.